# ROI Detector – Interactive Heatmap & Tile Explorer

This notebook demonstrates an end-to-end workflow for:

1. Loading a pretrained **ResNet-18** model for binary cytology classification (benign vs positive).
2. Building a **tissue mask** for a whole-slide image (WSI) based on HSV color thresholding.
3. Tiling the WSI into 224×224 patches and running inference on each tile.
4. Reconstructing a **probability heatmap** over the full WSI and exporting a slide-ready overlay.
5. Providing both:
   - A **static heatmap figure**, and  
   - An **interactive Gradio UI** where users can click the WSI heatmap and view the corresponding tiles and model predictions.

## 1. Environment Setup & Dependencies

In this section we install required libraries and set up the Colab environment.

- `timm` – model utilities (optional, but often useful)
- `gradio` – to build a user-friendly web UI for interactive tile exploration
- `torch`, `torchvision` – for model inference
- `opencv-python` (`cv2`) – for image processing
- `Pillow` – for image I/O

We also mount Google Drive to access the WSIs, model weights, and result CSVs.

In [ ]:
!pip install timm -q
!pip install gradio -q

import os, json
import numpy as np
import pandas as pd

from PIL import Image
Image.MAX_IMAGE_PIXELS = None  # allow very large WSIs

import cv2
import torch
import torch.nn as nn
from torchvision import transforms, models

import matplotlib.pyplot as plt
from scipy.spatial import KDTree

from google.colab import drive
import gradio as gr

# Mount Google Drive
drive.mount('/content/drive')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("DEVICE =", DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEVICE = cuda


## 2. Model Loading (Best Fold Selection)

Here we:

1. Define the normalization statistics used during training.
2. Recreate the **ResNet-18** model architecture with a binary classification head.
3. Load the **best-performing fold** from a k-fold cross-validation summary CSV.
4. Load the corresponding `.pth` weight file and set the model to evaluation mode.

In [ ]:
# Normalization values taken from the training pipeline
train_mean = [0.6760896444, 0.4603109061, 0.6932405829]
train_std  = [0.1372977793, 0.1647925227, 0.1199045330]

def create_model(num_classes: int = 2):
    """Create a ResNet-18 model with a binary classification head."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

# Paths to results summary and model directory
RESULTS_CSV = '/content/drive/MyDrive/AIMLGroup18/Ekyaalo_Binary_Classifier/kfold_results_summary.csv'
MODEL_DIR   = '/content/drive/MyDrive/AIMLGroup18/Ekyaalo_CNN/Model_Weights/'

# Select the best fold by accuracy
df = pd.read_csv(RESULTS_CSV)
best_fold = int(df.loc[df['accuracy'].idxmax(), 'fold']) + 1
MODEL_PATH = os.path.join(MODEL_DIR, f'best_fold_{best_fold}.pth')

print("Best fold:", best_fold)
print("Model path:", MODEL_PATH)

# Instantiate and load weights
model = create_model()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval();

print("Model loaded and set to eval mode.")

Best fold: 1
Model path: /content/drive/MyDrive/AIMLGroup18/Ekyaalo_CNN/Model_Weights/best_fold_1.pth
Model loaded and set to eval mode.


## 3. WSI Preprocessing: Tissue Mask & Tiling

In this section we:

1. Build a **binary tissue mask** that highlights purple-stained cytology regions and suppresses the glass/background.
2. Tile the WSI into overlapping 224×224 patches, keeping only tiles with sufficient tissue coverage.
3. Define the preprocessing pipeline and run model inference to obtain per-tile probabilities.

In [ ]:
def build_wsi_tissue_mask(path: str):
    """Build a binary mask where purple cytology smear = 1, background = 0."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f'Cannot read: {path}')

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    # Hue: purple/blue range, Saturation: not too low, Value: not too dark
    lower = np.array([110, 40, 20], dtype=np.uint8)
    upper = np.array([170, 255, 255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower, upper)

    # Clean mask with morphology
    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return (mask > 0).astype(np.float32)


def tile_wsi_masked(path: str, wsi_mask: np.ndarray, tile: int = 224, overlap: int = 112):
    """
    Tile the WSI and keep tiles with at least 10% purple tissue coverage.

    Args:
        path: Path to the WSI.
        wsi_mask: Binary tissue mask (H × W).
        tile: Tile size in pixels.
        overlap: Overlap between adjacent tiles.

    Returns:
        tiles: List of PIL.Image tiles.
        coords: List of (x, y) top-left coordinates for each tile.
        size: (W, H) of the original WSI.
    """
    img = Image.open(path).convert('RGB')
    W, H = img.size
    stride = tile - overlap

    tiles, coords = [], []
    for y in range(0, H - tile, stride):
        for x in range(0, W - tile, stride):
            if np.mean(wsi_mask[y:y+tile, x:x+tile]) < 0.10:
                continue
            tiles.append(img.crop((x, y, x+tile, y+tile)))
            coords.append((x, y))
    return tiles, coords, (W, H)


# CNN preprocessing transform (must mirror training)
cnn_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])


def infer_tiles(model, tiles):
    """Run model inference on each tile and return malignant probabilities."""
    probs = []
    for t in tiles:
        x = cnn_tf(t).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(model(x), dim=1)[0, 1].item()
        probs.append(p)
    return np.array(probs, dtype=np.float32)

## 4. Heatmap Reconstruction & Slide-Ready Overlay

Once we have per-tile probabilities, we:

1. Reconstruct a **dense probability heatmap** over the full WSI canvas.
2. Smooth and re-apply the tissue mask.
3. Create a **composite overlay** of the original WSI and the heatmap for slide or figure export.

In [ ]:
def build_heatmap(coords, probs, size, wsi_mask, tile: int = 224):
    """Reconstruct a dense heatmap from tile probabilities."""
    W, H = size
    heat = np.zeros((H, W), dtype=np.float32)

    # Paint each tile's probability into the heatmap
    for (x, y), p in zip(coords, probs):
        heat[y:y+tile, x:x+tile] = p

    # Apply tissue mask, smooth, and re-mask
    heat *= wsi_mask
    heat = cv2.GaussianBlur(heat, (0, 0), sigmaX=6, sigmaY=6)
    heat *= wsi_mask

    return heat


def export_slide_overlay(wsi_path, heat, alpha: float = 0.45, out: str = "SLIDE_OUTPUT.png"):
    """Create and save a slide-ready WSI + heatmap overlay image."""
    base = cv2.cvtColor(cv2.imread(wsi_path), cv2.COLOR_BGR2RGB)

    # Normalize heatmap into [0, 255] and colorize
    heat_norm = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    heat_color = cv2.applyColorMap((heat_norm * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat_color = cv2.cvtColor(heat_color, cv2.COLOR_BGR2RGB)

    # Only overlay where heatmap is non-zero
    mask = (heat_norm > 0.05).astype(np.float32)
    mask3 = np.repeat(mask[:, :, None], 3, axis=2)

    overlay = (base * (1 - mask3) +
               (base * (1 - alpha) + heat_color * alpha) * mask3).astype(np.uint8)

    Image.fromarray(overlay).save(out)
    print("Slide exported →", out)
    return overlay

## 5. Run the Full Pipeline on a WSI

Here we tie everything together:

1. Specify the path to the WSI.
2. Build the tissue mask.
3. Tile the slide and run inference.
4. Build the probability heatmap.
5. Export a slide-ready heatmap overlay.
6. Prepare a downscaled version of the overlay for fast display and interactive use.

In [ ]:
# Path to the WSI
WSI_PATH = '/content/drive/MyDrive/AIMLGroup18/Ekyaalo_ROI_Detection/WSI/c1102-23-1.tif'

print("Building tissue mask...")
wsi_mask = build_wsi_tissue_mask(WSI_PATH)

print("Tiling WSI...")
tiles, coords, size = tile_wsi_masked(WSI_PATH, wsi_mask)
print("Number of tiles:", len(tiles))

print("Running inference on tiles...")
probs = infer_tiles(model, tiles)

print("Building heatmap...")
heat = build_heatmap(coords, probs, size, wsi_mask)

print("Exporting slide overlay...")
overlay_full = export_slide_overlay(WSI_PATH, heat)

# Downscale overlay for visualization and interaction
DISPLAY_MAX = 2500  # maximum display dimension
scale = DISPLAY_MAX / max(overlay_full.shape[:2])
overlay_display = cv2.resize(overlay_full, None, fx=scale, fy=scale)

# Build KD-tree of tile centers in full-resolution coordinates
tile_centers = [(x + 112, y + 112) for (x, y) in coords]
tree = KDTree(tile_centers)

print("Full overlay shape:", overlay_full.shape)
print("Display overlay shape:", overlay_display.shape)
print("Scale factor:", scale)

Building tissue mask...


## 6. Static Heatmap Visualization (Figure-Style)

This cell produces a static, high-resolution figure of the WSI + heatmap overlay.  
You can export this figure directly into slides, reports, or manuscripts.

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(overlay_display, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Interactive Gradio ROI Explorer

Finally, we wrap everything in a **Gradio interface** that allows users to:

- **Click** anywhere on the WSI heatmap.
- Retrieve all tiles within a configurable radius of the click.
- View each tile with its **benign/positive prediction** and probability.

This creates a user-friendly, browser-based UI on top of the pipeline above.

In [ ]:
def get_tiles_from_click(display_x, display_y, radius: int = 1500):
    """
    Given click coordinates (in display space), return nearby tiles.

    Args:
        display_x, display_y: Coordinates on the downscaled overlay_display image.
        radius: Search radius in full-resolution pixels.

    Returns:
        List of (tile_image_array, probability, (x, y)) for each tile.
    """
    # Convert from display coordinates to full-resolution WSI coordinates
    full_x = display_x / scale
    full_y = display_y / scale

    idxs = tree.query_ball_point([full_x, full_y], r=radius)

    results = []
    for idx in idxs:
        tile = tiles[idx]          # PIL image (224×224)
        px, py = coords[idx]       # top-left tile coordinate in full-res space
        p = probs[idx]             # malignant probability

        tile_np = np.array(tile)   # convert to NumPy array for Gradio
        results.append((tile_np, float(p), (px, py)))

    return results


def on_click(evt: gr.SelectData):
    """Gradio callback: called when the user clicks on the WSI image."""
    x, y = evt.index  # (x, y) in display coordinates

    results = get_tiles_from_click(x, y)

    if len(results) == 0:
        return [], "No tiles found near this region."

    gallery_tiles = [img for img, p, loc in results[:12]]

    # Build a nicely formatted description of the predictions
    lines = []
    for img, p, (px, py) in results[:12]:
        label = "POSITIVE" if p >= 0.5 else "BENIGN"
        lines.append(f"{label} (p = {p:.3f}) at tile origin ({px}, {py})")

    metadata_text = "\n".join(lines)
    return gallery_tiles, metadata_text


# Prepare the display image for Gradio (RGB)
overlay_rgb = cv2.cvtColor(overlay_display, cv2.COLOR_BGR2RGB)
overlay_pil = Image.fromarray(overlay_rgb)


with gr.Blocks() as demo:
    gr.Markdown("# Slide Image Annotator")
    gr.Markdown("""
    Click on the heatmap to explore model predictions for regions of interest.

    - The image on the left is a downscaled WSI + heatmap overlay.
    - The gallery on the right shows 224×224 tiles near the click location.
    - Each tile is labeled as **POSITIVE** or **BENIGN** with its probability.
    """)

    with gr.Row():
        img = gr.Image(
            value=overlay_pil,
            label="WSI + Heatmap Overlay (clickable)",
            interactive=True
        )

    with gr.Row():
        gallery = gr.Gallery(
            label="Extracted Tiles (Nearest to Click)",
            columns=4,
            height="auto"
        )
        text_output = gr.Textbox(
            label="Tile Predictions",
            lines=12
        )

    # Wire the click event to our callback
    img.select(on_click, outputs=[gallery, text_output])


demo.launch(debug=True, share=True)